# 循環神經網路

在前一個單元中，我們涵蓋了豐富的語意表示文字。我們所使用的架構捕捉了句子中單字的綜合意義，但它並未考慮到單字的<strong>順序</strong>，因為嵌入之後的聚合操作從原始文本中去除了這些資訊。由於這些模型無法表示單字的排序，它們就無法解決較複雜或具歧義性的任務，例如文本生成或問答。

為了捕捉文本序列的意義，我們將使用一種稱為<strong>循環神經網路</strong>（recurrent neural network，RNN）的神經網路架構。使用 RNN 時，我們會將句子逐一通過網路的每個字元，並由網路產生一些<strong>狀態</strong>，接著將該狀態與下一個字元一起傳回網路。

![顯示循環神經網路生成的範例圖。](../../../../../translated_images/zh-TW/rnn.27f5c29c53d727b5.webp)

給定輸入的字元序列 $X_0,\dots,X_n$，RNN 建立一系列神經網路區塊，並使用反向傳播進行端對端訓練。每個網路區塊以一對 $(X_i,S_i)$ 作為輸入，並產生 $S_{i+1}$ 作為結果。最終狀態 $S_n$ 或輸出 $Y_n$ 會傳入線性分類器以產生結果。所有網路區塊共用相同權重，並透過一次反向傳播通過端對端地被訓練。

> 上圖展示了展開形式（左側）的循環神經網路，以及較緊湊的循環表示法（右側）。重要的是要理解所有 RNN 單元都具有相同的<strong>共享權重</strong>。

因為狀態向量 $S_0,\dots,S_n$ 會透過網路傳遞，RNN 能學習單字之間的序列依賴性。例如，當序列中出現單字 *not* 時，它可以學會將狀態向量中的某些元素反向。

在內部，每個 RNN 單元包含兩個權重矩陣：$W_H$ 與 $W_I$，以及偏差 $b$。每次 RNN 步驟中，給定輸入 $X_i$ 和輸入狀態 $S_i$，輸出狀態依照 $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$ 計算，其中 $f$ 是激活函數（通常是 $\tanh$）。

> 針對像是文本生成（我們會在下一個單元討論）或機器翻譯這類問題，我們同時也希望在每個 RNN 步驟中取得輸出值。這種情況下，還有另一個矩陣 $W_O$，輸出計算為 $Y_i=f(W_O\times S_i+b_O)$。

讓我們看看循環神經網路如何幫助我們對新聞資料集進行分類。

> 對於沙盒環境，我們需要執行下方程式碼以確保所需的函式庫已安裝，並且資料已預先讀取。如果您是在本機執行，可以跳過下方的程式碼。


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

在訓練大型模型時，GPU 記憶體配置可能會成為一個問題。我們也可能需要嘗試不同的小批次大小，以便數據能夠適合我們的 GPU 記憶體，同時訓練速度也足夠快。如果您在自己的 GPU 機器上運行此代碼，可以嘗試調整小批次大小以加速訓練。

> <strong>注意</strong>：某些版本的 NVidia 驅動程式已知在訓練模型後不會釋放記憶體。我們在此筆記本中運行了多個範例，在某些設置中可能會導致記憶體耗盡，特別是當您在同一筆記本中進行自己的實驗時。如果在開始訓練模型時遇到奇怪的錯誤，您可能需要重新啟動筆記本內核。


In [3]:
batch_size = 16
embed_size = 64

## 簡單 RNN 分類器

對於簡單的 RNN，每個循環單元都是一個簡單的線性網路，它接收輸入向量和狀態向量，並產生新的狀態向量。在 Keras 中，這可以用 `SimpleRNN` 層來表示。

雖然我們可以直接將獨熱編碼的標記傳入 RNN 層，但由於其高維度，這並不是一個好主意。因此，我們將使用嵌入層來降低詞向量的維度，接著是 RNN 層，最後是 `Dense` 分類器。

> <strong>注意</strong>：在維度不是很高的情況下，例如使用字符級標記化時，將獨熱編碼的標記直接傳入 RNN 單元可能是合理的。


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **注意：** 我們這裡使用未訓練過的嵌入層以簡化處理，但為了獲得更好的結果，我們可以使用前面單元中描述的 Word2Vec 預訓練嵌入層。將此程式碼改寫為使用預訓練嵌入層將是一個很好的練習。

現在讓我們來訓練 RNN。一般來說，訓練 RNN 是相當困難的，因為當 RNN 單元沿著序列長度展開時，反向傳播中涉及的層數會非常多。因此我們需要選擇較小的學習率，並在較大的資料集上訓練網路以取得良好結果。這可能會花費相當長的時間，因此建議使用 GPU。

為了加快速度，我們將只在新聞標題上訓練 RNN 模型，省略新聞描述。你也可以嘗試帶描述一起訓練，看看是否能使模型成功訓練。


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> <strong>注意</strong> 這裡的準確度可能較低，因為我們僅在新聞標題上進行訓練。


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


現在我們正在使用遮罩，可以在整個標題和描述數據集上訓練模型。

> <strong>注意</strong>：你是否注意到我們一直在使用一個在新聞標題上訓練的向量化器，而不是整篇文章的內容？這可能導致某些標記被忽略，因此最好重新訓練向量化器。不過，這可能只有非常小的影響，所以為了簡單起見，我們將堅持使用先前預訓練的向量化器。


## LSTM：長短期記憶

RNN 的主要問題之一是 <strong>梯度消失</strong>。RNN 可能相當長，在反向傳播時，梯度可能很難一路傳回到網路的第一層。當發生這種情況時，網路無法學習遠距標記間的關係。避免這個問題的一種方法是使用 <strong>門控</strong> 來引入 <strong>明確的狀態管理</strong>。引入門控的兩種最常見架構是 <strong>長短期記憶</strong> (LSTM) 和 <strong>門控結合單元</strong> (GRU)。我們這裡將涵蓋 LSTM。

![顯示一個長短期記憶單元範例的圖片](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

LSTM 網路的組成方式與 RNN 類似，但有兩種狀態會從一層傳遞到下一層：實際狀態 $c$ 和隱藏向量 $h$。在每個單元中，隱藏向量 $h_{t-1}$ 與輸入 $x_t$ 結合，並透過 <strong>門控</strong> 共同控制狀態 $c_t$ 和輸出 $h_t$ 的變化。每個門控都使用 sigmoid 激活函數（輸出範圍為 $[0,1]$），可視為與狀態向量相乘時的逐位遮罩。LSTM 擁有以下門控（如上圖從左到右）：
* <strong>遺忘門</strong>，決定需要忘記狀態向量 $c_{t-1}$ 的哪些成分，哪些成分需保持。
* <strong>輸入門</strong>，決定要從輸入向量和先前隱藏向量中融入多少資訊到狀態向量中。
* <strong>輸出門</strong>，根據新的狀態向量決定哪些成分將用來產生新的隱藏向量 $h_t$。

狀態 $c$ 的各個成分可視為可以切換開關的旗標。例如，當序列中出現名字 *Alice* 時，我們猜測它指的是女性，因此會在狀態中把表示句子有女性名詞的旗標打開。當後續遇到詞語 *and Tom* 時，我們會把表示有複數名詞的旗標打開。透過操控狀態，我們便能追蹤句子的語法特性。

> <strong>注意</strong>：這裡有個很棒的資源，能幫助理解 LSTM 內部機制：[Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/)，作者是 Christopher Olah。

雖然 LSTM 單元的內部結構看似複雜，但 Keras 把這個實作隱藏在 `LSTM` 層中，因此在上面範例中，我們唯一需要做的就是替換循環層：


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> <strong>注意</strong> 訓練 LSTM 也相當緩慢，在訓練初期你可能不會看到明顯的準確度提升。你可能需要持續訓練一段時間才能達到良好的準確度。


## 雙向與多層 RNN

迄今為止，在我們的範例中，遞迴網路都是從序列的起點運行到序列的終點。這讓我們感覺很自然，因為這與我們閱讀或聆聽語音的方向相同。然而，對於需要隨機存取輸入序列的場景，讓遞迴運算在兩個方向上執行會更有意義。允許在兩個方向上計算的 RNN 被稱為<strong>雙向</strong> RNN，並且可以透過用特殊的 `Bidirectonal` 層包裹遞迴層來建立。

> <strong>注意</strong>：`Bidirectional` 層會將其內部的層複製兩份，並將其中一份的 `go_backwards` 屬性設定為 `True`，使其沿序列向相反方向進行。

遞迴網路，不論是單向或雙向，都能捕捉序列中的模式，並將其存入狀態向量或作為輸出返回。與卷積網路一樣，我們也可以在第一層之後建立另外一個遞迴層，以捕捉由第一層抽取的較低層次模式所組成的較高層次模式。這引出了<strong>多層 RNN</strong>的概念，它由兩層或更多的遞迴網路組成，其中前一層的輸出作為下一層的輸入。

![展示多層長短期記憶RNN的圖片](../../../../../translated_images/zh-TW/multi-layer-lstm.dd975e29bb2a59fe.webp)

*圖片取自 Fernando López [這篇精彩的文章](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3)。*

Keras 讓構建這類網路變得簡單，因為你只需要往模型中添加更多遞迴層。對於除最後一層以外的所有層，我們需要指定 `return_sequences=True` 參數，因為我們需要該層返回所有中間狀態，而不僅僅是遞迴計算的最終狀態。

現在來為我們的分類問題建立一個雙層的雙向 LSTM。

> <strong>注意</strong>，這段程式碼的執行時間依然相當久，但它為我們帶來迄今為止最高的準確率。因此或許值得等待並觀察結果。


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## 其他任務的 RNN

到目前為止，我們主要專注於使用 RNN 來分類文字序列。但它們可以處理更多任務，例如文字生成和機器翻譯──我們會在下一單元討論這些任務。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
